In [0]:
%sql
create schema if not exists streaming_orders.gold;

In [0]:
%sql
create schema if not exists streaming_orders.silver;
create schema if not exists streaming_orders.bronze;

Data Ingestion (Bronze Layer)

In [0]:
df = spark.read.json(
    "/Volumes/streaming_orders/bronze/bronze_orders/orders_bronze.json"
)

print("Records:", df.count())

df.write.mode("overwrite").saveAsTable(
    "streaming_orders.bronze.orders_bronze"
)

Records: 188


Read Bronze table

In [0]:
from pyspark.sql.functions import col

bronze_df = spark.table(
    "streaming_orders.bronze.orders_bronze"
)

print("Bronze Records:", bronze_df.count())

Bronze Records: 188


Data Quality Validation

In [0]:
silver_df = bronze_df.filter(
    (col("customer_name").isNotNull()) &
    (col("product").isNotNull()) &
    (col("amount") > 0) &
    (col("payment_status").isin("SUCCESS","FAILED"))
)

print("Silver Records:", silver_df.count())
display(silver_df)

Silver Records: 28


amount,customer_name,order_id,payment_status,product,timestamp
41012,Stephen Bailey,1167,FAILED,Keyboard,2026-06-01 16:31:53.851696
15916,Paula Key,9169,SUCCESS,Laptop,2026-06-01 16:31:57.857880
22066,Michael Williams,4744,SUCCESS,Laptop,2026-06-01 16:32:05.878586
23622,James Everett,2308,FAILED,Keyboard,2026-06-01 16:32:17.904505
38012,Amanda Ward,2513,SUCCESS,Laptop,2026-06-01 16:32:19.911269
24147,Terrence Harper,9882,FAILED,Mouse,2026-06-01 16:32:33.944327
20246,Christopher Buck,5910,FAILED,Laptop,2026-06-01 16:32:45.973102
29923,William Allen,6145,SUCCESS,Keyboard,2026-06-01 16:33:28.083628
1744,Anthony Reed,2263,SUCCESS,Mouse,2026-06-01 16:33:40.118234
36038,Crystal Ramirez,6251,FAILED,Keyboard,2026-06-01 16:33:46.133447


In [0]:
silver_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.silver.orders_silver"
    )

Quarantine Layer

In [0]:
from pyspark.sql.functions import when

quarantine_df = bronze_df.filter(
    (col("customer_name").isNull()) |
    (col("product").isNull()) |
    (col("amount") <= 0) |
    (~col("payment_status").isin("SUCCESS","FAILED"))
)

In [0]:
quarantine_df = quarantine_df.withColumn(
    "error_reason",
    when(col("customer_name").isNull(),"Missing Customer")
    .when(col("product").isNull(),"Missing Product")
    .when(col("amount") <= 0,"Invalid Amount")
    .when(
        ~col("payment_status").isin("SUCCESS","FAILED"),
        "Invalid Payment Status"
    )
)

In [0]:
quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.bronze.orders_quarantine"
    )

In [0]:
%sql
SELECT error_reason, COUNT(*)
FROM streaming_orders.bronze.orders_quarantine
GROUP BY error_reason;

error_reason,COUNT(*)
Invalid Payment Status,43
Missing Customer,28
Missing Product,50
Invalid Amount,39


Silver Layer

In [0]:
from pyspark.sql.functions import sum

gold_df = silver_df.groupBy("product") \
    .agg(sum("amount").alias("total_revenue"))

gold_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.gold.product_revenue"
    )

In [0]:
%sql
SELECT *
FROM streaming_orders.gold.product_revenue
ORDER BY total_revenue DESC;

product,total_revenue
Laptop,253189
Phone,165416
Mouse,153327
Keyboard,130595
Monitor,29159


In [0]:
%sql
SELECT COUNT(*)
FROM streaming_orders.silver.orders_silver;

COUNT(*)
28


Gold Layer - Product Revenue

In [0]:
from pyspark.sql.functions import sum

silver_df = spark.table(
    "streaming_orders.silver.orders_silver"
)

product_revenue_df = silver_df.groupBy("product") \
    .agg(
        sum("amount").alias("total_revenue")
    )

product_revenue_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.gold.product_revenue"
    )

In [0]:
%sql
SELECT *
FROM streaming_orders.gold.product_revenue
ORDER BY total_revenue DESC;

product,total_revenue
Laptop,253189
Phone,165416
Mouse,153327
Keyboard,130595
Monitor,29159


In [0]:
from pyspark.sql.functions import count

status_summary_df = silver_df.groupBy(
    "payment_status"
).agg(
    count("*").alias("order_count")
)

status_summary_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.gold.order_status_summary"
    )

In [0]:
%sql
SELECT *
FROM streaming_orders.gold.order_status_summary;

payment_status,order_count
SUCCESS,19
FAILED,9


In [0]:
from pyspark.sql.functions import to_date, sum

daily_revenue_df = silver_df \
    .withColumn("order_date", to_date("timestamp")) \
    .groupBy("order_date") \
    .agg(
        sum("amount").alias("daily_revenue")
    )

daily_revenue_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "streaming_orders.gold.daily_revenue"
    )

In [0]:
%sql
SELECT *
FROM streaming_orders.gold.daily_revenue;

order_date,daily_revenue
2026-06-01,731686
